# Global Automotive Industry Analysis: Data Cleaning & Preparation

## Project Overview
This notebook focuses on the initial phase of our global automotive industry analysis. Our goal is to clean, reshape, and prepare various datasets related to vehicle production, sales, and Electric Vehicle (EV) adoption. 

The cleaned data will serve as the foundation for our exploratory analysis and visualizations in Tableau, specifically addressing the transition from Internal Combustion Engine (ICE) vehicles to EVs.

### Key Objectives:
1.  **Standardize** column names and remove annotation noise (e.g., footnotes, references).
2.  **Reshape** wide-format data into a long format suitable for time-series analysis.
3.  **Handle Missing Data**: Identify and address gaps in the datasets.
4.  **Enrichment**: Patch automated results with manually researched and AI-assisted data improvements.
5.  **Consolidate**: Prepare final tables for visualization.

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Setup paths
DATA_RAW = Path('../data/raw')
DATA_INTERIM = Path('../data/interim')
DATA_PROCESSED = Path('../data/processed/Cleaned_Tables')
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

def strip_refs(x):
    """Utility to remove reference noise like [75], (2021), etc."""
    if isinstance(x, str):
        # Remove [digits], [letters], and content in parentheses
        x = re.sub(r"\[.*?\]|\(.*?\)", "", x)
        return x.replace("—", "").strip()
    return x

## 1. Global Vehicle Production Data

We merge multiple annual production tables, clean the country names, and pivot to a long format for year-over-year comparison.

In [ ]:
production_path = DATA_RAW / 'List of countries by motor vehicle production'
prod_files = sorted(production_path.glob('table_*.csv'))

dfs = [pd.read_csv(f) for f in prod_files]

# Clean headers
for df in dfs:
    df.columns = [re.sub(r"\[.*?\]", "", col).strip() for col in df.columns]

# Merge tables on Country
from functools import reduce
prod_wide = reduce(lambda left, right: pd.merge(left, right, on="Country", how="outer"), dfs)

# Melt to long format
year_cols = [c for c in prod_wide.columns if c.isdigit()]
prod_long = prod_wide.melt(id_vars="Country", value_vars=year_cols, var_name="Year", value_name="Production")

# Final cleaning
prod_long["Country"] = prod_long["Country"].apply(strip_refs)
prod_long["Year"] = prod_long["Year"].astype(int)
prod_long["Production"] = pd.to_numeric(prod_long["Production"].astype(str).str.replace(",", ""), errors="coerce")

# Filter and drop NaNs
prod_long_auto = prod_long[prod_long["Year"] >= 2018].dropna(subset=["Production"])
prod_long_auto.head()

## 2. EV Market Share (Stock & Sales)

Processing EV adoption data involves converting string percentages to floats and calculating global market shares.

In [ ]:
ev_path = DATA_RAW / 'Electric car use by country'
ev_stock = pd.read_csv(ev_path / 'table_2.csv')

ev_stock.columns = ["Region", "PEV_stock_23", "Annual_sales_23", "Market_share_23", "Share_of_cars_in_use_23"]
ev_stock = ev_stock.applymap(strip_refs)

# Numeric conversion
for col in ["PEV_stock_23", "Annual_sales_23"]:
    ev_stock[col] = pd.to_numeric(ev_stock[col].astype(str).str.replace(",", ""), errors="coerce")

for col in ["Market_share_23", "Share_of_cars_in_use_23"]:
    ev_stock[col] = pd.to_numeric(ev_stock[col].astype(str).str.replace("%", ""), errors="coerce") / 100

ev_stock_auto = ev_stock.copy()
ev_stock_auto["%_ev_sales_global"] = ev_stock_auto["Annual_sales_23"] / 13800000 # 2023 baseline
ev_stock_auto.head()

## 3. Historical EV Market Share Trends

Converting wide-format historical percentages into a clean time-series.

In [ ]:
ev_trends = pd.read_csv(ev_path / 'table_3.csv')
ev_trends.columns = [re.sub(r"\[.*?\]", "", col).strip() for col in ev_trends.columns]

# Clean and Melt
ev_trends = ev_trends.applymap(strip_refs)
for col in ev_trends.columns:
    if col != "Country":
        ev_trends[col] = pd.to_numeric(ev_trends[col].astype(str).str.replace("%", ""), errors="coerce")

ev_trends_long = ev_trends.melt(id_vars="Country", var_name="Year", value_name="EV_market_share")
ev_trends_long["Year"] = pd.to_numeric(ev_trends_long["Year"], errors="coerce")
ev_trends_long_auto = ev_trends_long[ev_trends_long["Year"] >= 2018].dropna()
ev_trends_long_auto.head()

## 4. Vehicles Per Capita

Merging population-adjusted vehicle data.

In [ ]:
vpc_path = DATA_RAW / 'List of countries and territories by motor vehicles per capita'
vpc = pd.read_csv(vpc_path / 'table_0.csv')

vpc = vpc.drop(columns=["Ref"], errors="ignore")
vpc = vpc.rename(columns={"Vehicles per 1,000 people": "vehicles_per_1000_people", "Region": "Country"})
vpc_auto = vpc[vpc["Year"] >= 2018]
vpc_auto.head()

## 5. Vehicle Export & Trade Data

Calculating trade balances and cleaning currency values.

In [ ]:
trade_path = DATA_RAW / 'List of countries by vehicle exports'
trade_24 = pd.read_csv(trade_path / 'table_0.csv')
trade_23 = pd.read_csv(trade_path / 'table_1.csv')

trade_24 = trade_24.rename(columns={
    "Value exported (thousands USD)": "Value_exported_2024",
    "Value imported (thousands USD)": "Value_imported_2024"
})

for col in ["Value_exported_2024", "Value_imported_2024"]:
    trade_24[col] = pd.to_numeric(trade_24[col].astype(str).str.replace(",", "").str.replace("−", "-"), errors="coerce")

trade_24_auto = trade_24.copy()
trade_24_auto["Trade_balance_2024"] = trade_24_auto["Value_exported_2024"] - trade_24_auto["Value_imported_2024"]
trade_24_auto.head()

## 6. Manual Enrichment & AI-Assisted Patching

In this final step, we load the manually enriched data (containing human-in-the-loop corrections and AI-filled missing values) from the `data/interim/` directory. 

We use the automated results as a base but prioritize the manual edits where they exist using the `combine_first` method. This ensures that any manual research performed on the datasets is preserved and applied on top of the automated cleaning logic.

In [ ]:
def enrich_data(auto_df, filename, join_cols):
    """Helper to patch automated data with manually enriched data."""
    interim_file = DATA_INTERIM / filename
    if interim_file.exists():
        enriched_df = pd.read_csv(interim_file)
        # Set index to join columns for proper alignment
        auto_df = auto_df.set_index(join_cols)
        enriched_df = enriched_df.set_index(join_cols)
        
        # combine_first takes values from enriched_df first, then auto_df
        final_df = enriched_df.combine_first(auto_df).reset_index()
        print(f"Enriched {filename}: Applied manual patches to {len(enriched_df)} records.")
        return final_df
    else:
        print(f"Warning: No interim file found for {filename}. Using automated results.")
        return auto_df

# Apply enrichment to each table
prod_final = enrich_data(prod_long_auto, "vehicle_production.csv", ["Country", "Year"])
ev_2_final = enrich_data(ev_stock_auto, "ev_2.csv", ["Region"])
ev_3_final = enrich_data(ev_trends_long_auto, "ev_3.csv", ["Country", "Year"])
vpc_final = enrich_data(vpc_auto, "vehicles_per_capita1.csv", ["Country", "Year"])
trade_final = enrich_data(trade_24_auto, "trade_data.csv", ["Country"])

# Save final versions to Cleaned_Tables
prod_final.to_csv(DATA_PROCESSED / "vehicle_production.csv", index=False)
ev_2_final.to_csv(DATA_PROCESSED / "ev_2.csv", index=False)
ev_3_final.to_csv(DATA_PROCESSED / "ev_3.csv", index=False)
vpc_final.to_csv(DATA_PROCESSED / "vehicles_per_capita1.csv", index=False)
trade_final.to_csv(DATA_PROCESSED / "trade_data.csv", index=False)

print("\nSuccess: All final datasets saved to data/processed/Cleaned_Tables/")

## Conclusion

The data is now standardized, reshaped, and enriched with manual edits. All final outputs are saved in `data/processed/Cleaned_Tables/`. 

**Next Steps:**
- Import these CSVs into Tableau.
- Establish relationships between `vehicle_production`, `ev_3` (Market Share), and `vehicles_per_capita` using **Country** and **Year** as linking fields.
- Address the core hypothesis: *Is EV adoption correlated with a decline in total vehicle production/sales?*